# Route labels — the golden set over our 50K composition

**Input.** `TargetComposition` — the 50,000 rows the composition package
selected. That is the dataset; nothing here invents queries.

**Output.** `data/route_labels/labels.parquet`: one row per labelled query
carrying the *route label* — which of `dense_only`, `pure_rrf`, `sparse_only`
retrieved best for it. Plus the losing routes' scores, the outcome shape, and
the composition's own `slice` / `checkable` / `label_lane` columns so results
can be read per slice.

**How the label is decided.** All three routes are run, each ranking is scored
by the router objective, argmax wins. No LLM is asked which route is better:
dense-vs-sparse depends on the corpus vocabulary and its IDF, neither of which
is in the query, so a model reading only the query is being asked to predict
something that isn't a function of its input.

**Honest coverage warning.** Labeling a row needs its dataset's corpus indexed
*and* its qrels. The registry cache is `[query_id, text]` for all 21 datasets —
doc_ids and qrels were dropped on ingest — so a lane opens only when a
per-dataset snapshot lands on disk. Snapshots exist for **beir-nfcorpus**
(323 rows) and, per SPEC d38, **msmarco-passage-dev** (7,697 of 15,678 — the
dev qrels cover 49.1%). `RouteLabels` reports every remaining row as
`unlabelled` rather than omitting it; lifting the registry's queries-only
restriction is the blocker for the rest.

**Prerequisites**

```bash
docker compose up -d          # local Qdrant on :6333
```

In [1]:
%load_ext autoreload
%autoreload 2

## 1 — Setup

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from qdrant_client import QdrantClient
from qdrant_client.models import Distance

load_dotenv(".env") or load_dotenv("../.env")

DENSE_MODEL, DENSE_SIZE = "BAAI/bge-small-en-v1.5", 384
SPARSE_MODEL = "Qdrant/bm25"

client = QdrantClient(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


def show(df: pd.DataFrame) -> None:
    display(Markdown(df.to_markdown(index=False)))


print("qdrant collections:", [c.name for c in client.get_collections().collections])

qdrant collections: ['clerc_v5', 'composition_selection', 'nf', 'nfcorpus_routes', 'trec_dl']


## 2 — Load the composition

`TargetComposition().build()` returns the selection, reading `selection.parquet`
from disk when it exists — so this does not re-run the fill.

In [3]:
from composition import TargetComposition
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective

selection = TargetComposition().build()
print(f"selection: {len(selection):,} rows x {len(selection.columns)} cols")
show(selection.head(5))

labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
print("objective:", labels.objective.name)

selection: 50,000 rows x 7 cols


| dataset       | query_id   | checkable   | slice   | label_lane   | floors                                          | query                        |
|:--------------|:-----------|:------------|:--------|:-------------|:------------------------------------------------|:-----------------------------|
| beir-nfcorpus | PLAIN-1018 | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | DHA                          |
| beir-nfcorpus | PLAIN-1088 | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | ECMO                         |
| beir-nfcorpus | PLAIN-112  | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | Food Dyes and ADHD           |
| beir-nfcorpus | PLAIN-1320 | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | Harvard Physicians’ Study II |
| beir-nfcorpus | PLAIN-1398 | True        | A       | qrels        | ['id:number' 'id:shape_guess' 'marker:acronym'] | IGF-1                        |

objective: 0.7*HitRate@1+0.3*NDCG@10


## 3 — What can be labelled today

`unlabelled` means the row is in the composition but its dataset has no
indexed corpus or no qrels on disk. This table is the real progress bar for
the golden set, and it should be re-read after every dataset lands.

In [ ]:
coverage = labels.coverage()
show(coverage)
print(f"selected {coverage.selected.sum():,} | "
      f"labelled {coverage.labelled.sum():,} | "
      f"unlabelled {coverage.unlabelled.sum():,} "
      f"({coverage.unlabelled.sum() / coverage.selected.sum() * 100:.1f}%)")

## 4 — Index one dataset's corpus

Starting with `beir-nfcorpus`: 3,633 documents, small enough to index in full
with no corpus sampling, and it ships human judgments so the labels are
checkable. It is also richly judged — 38.2 judged docs per query, and 300 of its
323 queries have two or more relevant documents, which is the only regime where
the choice of objective can change a label at all.

**Not** reusing the existing `nf` collection: it holds 33,633 points — 3,633
nfcorpus documents mixed with 30,000 trec-dl passages — so nfcorpus queries
would be scored against a corpus that is 90% unrelated.

Dense and sparse live as two named vector slots on one collection, which is what
lets a single Qdrant call fuse them. Idempotent: re-running skips the upload.

In [6]:
from hybrid_search_rrf_dataset.indexer import (
    CorpusDocument, CorpusIndexer, EmbeddingCache, EmbeddingConfig,
)
from hybrid_search_rrf_dataset.retrieval import SnapshotDataset

DATASET = "beir-nfcorpus"        # the composition's key
COLLECTION = "nfcorpus_routes"

source = SnapshotDataset("nfcorpus", path="data")   # written by RetrievalDataset.save()
corpus = source.corpus()

dense_cfg = EmbeddingConfig(
    name="dense_base", model_id=DENSE_MODEL, kind="dense",
    size=DENSE_SIZE, distance=Distance.COSINE,
)
sparse_cfg = EmbeddingConfig(name="sparse_base", model_id=SPARSE_MODEL, kind="sparse")

indexer = CorpusIndexer(
    client, COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
indexer.ensure_collection()

if client.count(COLLECTION, exact=True).count >= len(corpus):
    print(f"{COLLECTION}: already indexed — skipping upload")
else:
    indexer.upload([CorpusDocument(**r) for r in corpus.to_dict("records")], batch_size=64)
print(f"{COLLECTION}: {client.count(COLLECTION, exact=True).count:,} points")

nfcorpus_routes: already indexed — skipping upload
nfcorpus_routes: 3,633 points


## 5 — The three routes

`DenseOnlyStrategy` and `SparseOnlyStrategy` keep their raw scores;
`PureRRFStrategy` is Qdrant-native RRF, matching production's `Fusion::Rrf`.

RRF fuses by *rank position*, which is exactly why it can lose on top-1: a doc
ranked first by dense and 40th by sparse loses to one ranked third by both.

In [7]:
from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy, PureRRFStrategy, SparseOnlyStrategy,
)

args = (client, COLLECTION, dense_cfg, sparse_cfg)
dense, hybrid, sparse = (
    DenseOnlyStrategy(*args), PureRRFStrategy(*args), SparseOnlyStrategy(*args)
)

probe = labels.rows_for(DATASET)["query"].iloc[0]
print(f"probe (a real selection row): {probe!r}\n")
for s in (dense, hybrid, sparse):
    top = list(s.rank(probe).items())[:3]
    print(f"  {s.name:12s} " + "  ".join(f"{d}={v:.3f}" for d, v in top))

probe (a real selection row): 'DHA'

  dense_only   MED-5095=0.735  MED-5091=0.718  MED-4936=0.714
  pure_rrf     MED-5095=1.000  MED-4936=0.583  MED-5091=0.533
  sparse_only  MED-5095=3.486  MED-4936=3.465  MED-839=3.449


## 6 — Label this dataset's selection rows

`RouteLabels.label` narrows the source to the composition's query ids via
`QuerySubset`, runs `GoldenRoutingBuilder`, and merges the result into
`labels.parquet` — replacing only this dataset's rows.

The objective is `0.7·HitRate@1 + 0.3·NDCG@10`. Because the hit weight exceeds
the NDCG weight the score ranges are disjoint (rank-1 hit ⇒ ≥0.700, miss ⇒
≤0.300), so it is lexicographic: top-1 decides, NDCG@10 only breaks ties within
each group. `min_relevance=1` because nfcorpus grades are 1 and 2 only.

In [8]:
labelled = labels.label(source, dense, hybrid, sparse, dataset=DATASET)
print(f"labelled {len(labelled):,} rows -> {labels.labels_path}")
show(labelled.head(8).round(3))

labelled 323 rows -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/labels.parquet


| dataset       | query_id   | route       | query                                                    |   score |   score_dense_only |   score_pure_rrf |   score_sparse_only | shape         | metric_name               |   min_relevance | slice   | checkable   | label_lane   |
|:--------------|:-----------|:------------|:---------------------------------------------------------|--------:|-------------------:|-----------------:|--------------------:|:--------------|:--------------------------|----------------:|:--------|:------------|:-------------|
| beir-nfcorpus | PLAIN-2    | pure_rrf    | Do Cholesterol Statin Drugs Cause Breast Cancer?         |   0.946 |              0.933 |            0.946 |               0.916 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-12   | pure_rrf    | Exploiting Autophagy to Live Longer                      |   0.773 |              0     |            0.773 |               0.773 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-23   | dense_only  | How to Reduce Exposure to Alkylphenols Through Your Diet |   0.847 |              0.847 |            0.08  |               0     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-33   | dense_only  | What’s Driving America’s Obesity Problem?                |   0.834 |              0.834 |            0.116 |               0.063 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-44   | pure_rrf    | Who Should be Careful About Curcumin?                    |   0.78  |              0.776 |            0.78  |               0.053 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-56   | dense_only  | Foods for Glaucoma                                       |   0.903 |              0.903 |            0.89  |               0.853 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-68   | dense_only  | What is Actually in Chicken Nuggets?                     |   0.914 |              0.914 |            0.907 |               0.859 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-78   | sparse_only | What Do Meat Purge and Cola Have in Common?              |   0.025 |              0     |            0     |               0.025 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |

## 7 — Outcome shapes

Only one of the three shapes teaches the router about dense-vs-sparse.

| shape | meaning |
| --- | --- |
| `routes_differ` | the quality signal — the trainable set |
| `all_tied` | any route works; serve the cheapest. Signal for the *speed* goal |
| `all_zero` | nothing relevant found by any route — unanswerable, and **no valid label exists**. The argmax falls through to list order and reports `dense_only`, which is fabricated |

Read per composition slice: A/B are span-evidence rows, C is the stat strata,
D is the feature-blind draw.

In [9]:
shapes = (labelled.groupby(["slice", "shape"]).size()
          .unstack(fill_value=0))
shapes["total"] = shapes.sum(axis=1)
show(shapes.reset_index())

overall = labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = labelled[labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

| slice   |   all_tied |   all_zero |   routes_differ |   total |
|:--------|-----------:|-----------:|----------------:|--------:|
| A       |          2 |          7 |              19 |      28 |
| B       |          0 |          0 |               8 |       8 |
| C       |         24 |         70 |             193 |     287 |

| shape         |   rows | share   |
|:--------------|-------:|:--------|
| routes_differ |    220 | 68.1%   |
| all_zero      |     77 | 23.8%   |
| all_tied      |     26 | 8.0%    |

route distribution over the 220 trainable rows:


| route       |   rows | share   |
|:------------|-------:|:--------|
| dense_only  |    107 | 48.6%   |
| pure_rrf    |     64 | 29.1%   |
| sparse_only |     49 | 22.3%   |

majority-class baseline: 49% (always predict dense_only)


## 8 — Constant-route baselines

The bar a router must clear is **the best constant route**, not random. A
candidate can post respectable regret against the oracle and still lose to one
line of code, so these belong in every comparison.

In [10]:
from hybrid_search_rrf_dataset.evaluation import compare
from hybrid_search_rrf_dataset.golden import BaselineBuilder, GoldenRoutingBuilder
from hybrid_search_rrf_dataset.retrieval import QuerySubset

subset = QuerySubset(source, labels.rows_for(DATASET)["query_id"])
oracle = GoldenRoutingBuilder(
    dense, hybrid, sparse, objective=labels.objective
).build_or_load(subset, path=f"data/route_labels/{DATASET}_oracle")

rows = []
for strategy in (dense, hybrid, sparse):
    const = BaselineBuilder(strategy, objective=labels.objective).build_or_load(
        subset, path=f"data/route_labels/{DATASET}_const_{strategy.name}"
    )
    s = compare(oracle, const)
    rows.append({
        "always pick": str(strategy.name),
        "mean score": round(s.mean_metric_candidate, 3),
        "oracle ceiling": round(s.mean_metric_golden, 3),
        "mean regret": round(s.mean_regret, 3),
        "p90 regret": round(s.p90_regret, 3),
        "reaches oracle": f"{s.oracle_hit_rate_pct:.0f}%",
        "route agreement": f"{s.route_agreement_pct:.0f}%",
    })
show(pd.DataFrame(rows))

| always pick   |   mean score |   oracle ceiling |   mean regret |   p90 regret | reaches oracle   | route agreement   |
|:--------------|-------------:|-----------------:|--------------:|-------------:|:-----------------|:------------------|
| dense_only    |        0.403 |            0.504 |         0.102 |        0.727 | 65%              | 65%               |
| pure_rrf      |        0.443 |            0.504 |         0.062 |        0.047 | 55%              | 20%               |
| sparse_only   |        0.389 |            0.504 |         0.116 |        0.739 | 53%              | 15%               |

## 9 — Objective sensitivity, at zero retrieval cost

`route_rankings` holds each route's top-10 doc ids and the judgments live in
`QrelStore`, so any metric depending only on the *order* of the top 10 can be
recomputed without touching Qdrant. That is what makes the objective a
reversible decision instead of a one-way door.

We synthesize descending scores from the stored order — HitRate@1 and NDCG@10
depend only on that order, so the recomputation is exact. It cannot evaluate a
metric needing rank 11+ or the raw retrieval scores.

In [11]:
from hybrid_search_rrf_dataset.objective import NDCGObjective
from hybrid_search_rrf_dataset.qrels import QrelStore

lookup = QrelStore.from_dataset(source).lookup(source.name)


def relabel(objective) -> pd.Series:
    picks = []
    for row in oracle:
        gold = lookup.get(row.query_id, {})
        scored = {
            route: objective.assess(
                {d: 1.0 / (i + 1) for i, d in enumerate(ids)}, gold
            )[0]
            for route, ids in row.route_rankings.items()
        }
        picks.append(max(scored, key=lambda route: scored[route]))
    return pd.Series(picks)


shipped = pd.Series([str(r.strategy_name) for r in oracle])
variants = {
    "0.7·HR@1 + 0.3·NDCG@10  (shipped)": RouterObjective(min_relevance=1),
    "0.5·HR@1 + 0.5·NDCG@10": RouterObjective(hit_weight=0.5, ndcg_weight=0.5, min_relevance=1),
    "bare NDCG@10 (no top-1 term)": NDCGObjective(min_relevance=1),
    "shipped, stricter min_relevance=2": RouterObjective(min_relevance=2),
}

rows = []
for name, objective in variants.items():
    picks = relabel(objective)
    rows.append({
        "objective": name,
        "flips": int((picks != shipped).sum()),
        "flip rate": f"{(picks != shipped).mean() * 100:.1f}%",
        **{f"picks {r}": int((picks == r).sum())
           for r in ("dense_only", "pure_rrf", "sparse_only")},
    })
show(pd.DataFrame(rows))

| objective                         |   flips | flip rate   |   picks dense_only |   picks pure_rrf |   picks sparse_only |
|:----------------------------------|--------:|:------------|-------------------:|-----------------:|--------------------:|
| 0.7·HR@1 + 0.3·NDCG@10  (shipped) |       0 | 0.0%        |                211 |               63 |                  49 |
| 0.5·HR@1 + 0.5·NDCG@10            |       0 | 0.0%        |                211 |               63 |                  49 |
| bare NDCG@10 (no top-1 term)      |       3 | 0.9%        |                212 |               62 |                  49 |
| shipped, stricter min_relevance=2 |      89 | 27.6%       |                293 |               16 |                  14 |

## 10 — Where the golden set stands

Re-read coverage now that one dataset is labelled, and project the trainable
yield. The projection assumes other datasets behave like this one, which they
will not — nfcorpus is a single 3,633-document medical corpus, unusually
richly judged. Treat it as an order of magnitude, not a forecast.

In [12]:
coverage = labels.coverage()
show(coverage[coverage.labelled > 0])

done = labels.load()
trainable = (done["shape"] == "routes_differ").mean()
print(f"labelled so far:        {len(done):,} of {len(selection):,} rows")
print(f"trainable share:        {trainable * 100:.1f}%")
print(f"projected yield at 50K: ~{int(trainable * len(selection)):,} rows "
      f"(extrapolated from one dataset — see caveat above)")
print()
print("next unblock, largest first:")
show(coverage[coverage.labelled == 0].head(5)[["dataset", "selected", "unlabelled"]])

| dataset       |   selected |   labelled |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:--------------|-----------:|-----------:|-------------:|----------------:|-----------:|-----------:|
| beir-nfcorpus |        323 |        323 |            0 |             220 |         26 |         77 |

labelled so far:        323 of 50,000 rows
trainable share:        68.1%
projected yield at 50K: ~34,055 rows (extrapolated from one dataset — see caveat above)

next unblock, largest first:


| dataset              |   selected |   unlabelled |
|:---------------------|-----------:|-------------:|
| orcas                |      15744 |        15744 |
| msmarco-passage-dev  |      15678 |        15678 |
| rarb-math            |       6276 |         6276 |
| crumb-code-retrieval |       3665 |         3665 |
| crumb-legal-qa       |       3550 |         3550 |

## 11 — msmarco-passage-dev: the composition's largest lane (SPEC d38)

15,678 composition rows — 31% of the 50K. The local dev qrels cover 7,697 of
them (49.1%); the rest stay `unlabelled` in coverage — a gap to report, never
a licence to substitute other queries. The gap is MS MARCO's own: the full
dev set ships 101,093 queries but judgments were released for only 55,578
(55%), and the fill drew judgment-blind — the composition's `checkable=True`
was assigned per-dataset, not per-query. Median **one** judged passage per
query against nfcorpus's 16, so this lane sits at the opposite end of the
judgment-density axis: the regime where two different top-10 lists cannot
both be right.

The corpus recipe (d38c): every judged-relevant passage for the selected
queries is force-included, then padded with uniform-random passages from the
full 8.8M collection to **100,000** total, fixed seed. Uniform sampling
preserves the collection's vocabulary/IDF profile — the d37(g) fix: trec-dl's
judged-docs-only corpus was near-all answers, which inflates dense and
starves sparse.

Materialization is one-time and local (the ir_datasets collection is already
on disk; first `docs_store` access builds its index). Re-runs read the
snapshot back via `SnapshotDataset`.

In [13]:
from pathlib import Path

from hybrid_search_rrf_dataset.retrieval import MSMarcoDev

MS_DATASET = "msmarco-passage-dev"      # composition key == source name here
MS_COLLECTION = "msmarco_routes"

if not (Path("data") / MS_DATASET / "corpus.parquet").exists():
    ms = MSMarcoDev(
        query_ids=labels.rows_for(MS_DATASET)["query_id"],
        corpus_size=100_000,
        seed=0,                          # d38(c): fixed seed, recipe is a parameter
    )
    ms.materialize()
    ms.save("data")

ms_source = SnapshotDataset(MS_DATASET, path="data")
ms_corpus, ms_queries, ms_qrels = ms_source.corpus(), ms_source.queries(), ms_source.qrels()
print(f"corpus  {len(ms_corpus):,} passages "
      f"({ms_qrels.doc_id.nunique():,} judged-relevant, rest uniform distractors)")
print(f"queries {len(ms_queries):,} of {len(labels.rows_for(MS_DATASET)):,} composition rows "
      f"({len(ms_queries) / len(labels.rows_for(MS_DATASET)) * 100:.1f}% have dev qrels)")
print(f"qrels   {len(ms_qrels):,} judgments, grades {sorted(ms_qrels.relevance.unique())}")

[INFO] [starting] building docstore                                  
docs_iter: 100%|█████████████████| 8841823/8841823 [00:32<00:00, 275879.56doc/s]
[INFO] [finished] docs_iter: [00:32] [8841823doc] [275877.21doc/s]   
[INFO] [finished] building docstore [32.05s]                         
[INFO] Opening /Users/andrei/.ir_datasets/msmarco-passage/collection.tsv.pklz4/bin with direct file access
docs:msmarco-passage-dev: 100%|██████████| 100000/100000 [00:41<00:00, 2395.05doc/s]


corpus  100,000 passages (8,219 judged-relevant, rest uniform distractors)
queries 7,697 of 15,678 composition rows (49.1% have dev qrels)
qrels   8,228 judgments, grades [np.int64(1)]


## 12 — Index the 100K corpus

Same two named vector slots as every route collection. The one-time cost is
the dense pass over 100K passages (order of an hour on this machine); the
embedding cache makes re-runs cheap, and the upload skips when the collection
is already full.

In [14]:
ms_indexer = CorpusIndexer(
    client, MS_COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
ms_indexer.ensure_collection()

if client.count(MS_COLLECTION, exact=True).count >= len(ms_corpus):
    print(f"{MS_COLLECTION}: already indexed — skipping upload")
else:
    ms_indexer.upload(
        [CorpusDocument(**r) for r in ms_corpus.to_dict("records")], batch_size=64
    )
print(f"{MS_COLLECTION}: {client.count(MS_COLLECTION, exact=True).count:,} points")

embed:sparse_base: 100%|██████████| 100000/100000 [00:04<00:00, 21181.53it/s]


msmarco_routes: 100,000 points


## 13 — Label the lane

Same argmax rule as the nfcorpus anchor (d38e), so the two datasets differ by
exactly one variable — the index they are scored on. `RouteLabels.label`
narrows the 15,678 selection rows to the snapshot's queries via `QuerySubset`
and merges only this dataset's rows into `labels.parquet`. `min_relevance=1`
holds: the dev qrels are binary.

~7,700 queries × 3 routes; at nfcorpus throughput this is on the order of ten
minutes against local Qdrant.

In [15]:
ms_args = (client, MS_COLLECTION, dense_cfg, sparse_cfg)
ms_dense, ms_hybrid, ms_sparse = (
    DenseOnlyStrategy(*ms_args), PureRRFStrategy(*ms_args), SparseOnlyStrategy(*ms_args)
)

ms_labelled = labels.label(ms_source, ms_dense, ms_hybrid, ms_sparse, dataset=MS_DATASET)
print(f"labelled {len(ms_labelled):,} rows -> {labels.labels_path}")
show(ms_labelled.head(8).round(3))

goldenroutingbuilder:msmarco-passage-dev: 100%|██████████| 7697/7697 [05:51<00:00, 21.87it/s]


labelled 7,697 rows -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/labels.parquet


| dataset             |   query_id | query                                                    | route      |   score |   score_dense_only |   score_pure_rrf |   score_sparse_only | shape         | metric_name               |   min_relevance | slice   | checkable   | label_lane   |
|:--------------------|-----------:|:---------------------------------------------------------|:-----------|--------:|-------------------:|-----------------:|--------------------:|:--------------|:--------------------------|----------------:|:--------|:------------|:-------------|
| msmarco-passage-dev |          2 | Androgen receptor define                                 | dense_only |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |    1048600 | what is patricia cornwell's latest book                  | dense_only |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | A       | True        | qrels        |
| msmarco-passage-dev |     524332 | treating tension headaches without medication            | dense_only |   0.189 |              0.189 |             0.15 |               0     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | A       | True        | qrels        |
| msmarco-passage-dev |    1048663 | what is palm harbor florida                              | dense_only |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |     786568 | what is price of pressure treated lumber 2x6x8           | dense_only |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |     786598 | what is primary and non-contributory under the liability | dense_only |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |    1048836 | who plays velma in scooby doo 2                          | dense_only |   1     |              1     |             1    |               0.189 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | A       | True        | qrels        |
| msmarco-passage-dev |    1048846 | what is option button style ?                            | dense_only |   1     |              1     |             1    |               0.189 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |

## 14 — Outcome shapes on a realistic index

Same three shapes as §7, now on a corpus that is 92% distractors. With a
median of **one** judged passage per query, `all_tied` requires all three
routes to place that same passage at the same effective rank, and
`routes_differ` means at least one route actually found it while another
did not — a much harder tie regime than nfcorpus's.

In [16]:
ms_shapes = (ms_labelled.groupby(["slice", "shape"]).size()
             .unstack(fill_value=0))
ms_shapes["total"] = ms_shapes.sum(axis=1)
show(ms_shapes.reset_index())

overall = ms_labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(ms_labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = ms_labelled[ms_labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

| slice   |   all_tied |   all_zero |   routes_differ |   total |
|:--------|-----------:|-----------:|----------------:|--------:|
| A       |        818 |         52 |             978 |    1848 |
| B       |        190 |         11 |             206 |     407 |
| C       |       1710 |         62 |            1971 |    3743 |
| D       |        820 |         28 |             851 |    1699 |

| shape         |   rows | share   |
|:--------------|-------:|:--------|
| routes_differ |   4006 | 52.0%   |
| all_tied      |   3538 | 46.0%   |
| all_zero      |    153 | 2.0%    |

route distribution over the 4,006 trainable rows:


| route       |   rows | share   |
|:------------|-------:|:--------|
| dense_only  |   3333 | 83.2%   |
| pure_rrf    |    474 | 11.8%   |
| sparse_only |    199 | 5.0%    |

majority-class baseline: 83% (always predict dense_only)


## 15 — Where the golden set stands, two datasets in

The open label-form question (argmax one-hot vs the per-route score vector,
TODOS) turns on the **margin** — winner's score minus runner-up's. A fat
margin is a fact about retrieval; a thin one is a coin toss the objective
happened to break, and it flips when the encoder or corpus changes.
`decisive` counts rows with margin ≥ 0.06 *and* a rank-1 hit — roomier than
any NDCG-tail wiggle, and excluding "least bad" wins where every route missed.

nfcorpus margins were thin because 86% of its corpus is judged relevant to
*something* — two disjoint top-10s can both be right. msmarco's
median-1-relevant regime is the counter-test: either a route surfaced the one
judged passage or it scored zero, so ties require actually retrieving the
same passage at the same rank.

In [17]:
SCORES = ["score_dense_only", "score_pure_rrf", "score_sparse_only"]

done = labels.load()
done["margin"] = done[SCORES].max(axis=1) - done[SCORES].apply(
    lambda r: sorted(r)[-2], axis=1
)
done["hit"] = done[SCORES].max(axis=1) >= 0.7

rows = []
for ds, group in done.groupby("dataset"):
    differ = group[group["shape"] == "routes_differ"]
    decisive = (differ.margin >= 0.06) & differ.hit
    rows.append({
        "dataset": ds,
        "labelled": len(group),
        "routes_differ": len(differ),
        "median margin": round(differ.margin.median(), 3),
        "p90 margin": round(differ.margin.quantile(0.9), 3),
        "decisive": int(decisive.sum()),
        "decisive share": f"{decisive.mean() * 100:.0f}%",
    })
show(pd.DataFrame(rows))

coverage = labels.coverage()
show(coverage[coverage.labelled > 0])
print(f"labelled {coverage.labelled.sum():,} of {coverage.selected.sum():,} "
      f"({coverage.labelled.sum() / coverage.selected.sum() * 100:.1f}%)")

| dataset             |   labelled |   routes_differ |   median margin |   p90 margin |   decisive | decisive share   |
|:--------------------|-----------:|----------------:|----------------:|-------------:|-----------:|:-----------------|
| beir-nfcorpus       |        323 |             220 |           0.011 |        0.707 |         28 | 13%              |
| msmarco-passage-dev |       7697 |            4006 |           0     |        0.811 |        927 | 23%              |

| dataset             |   selected |   labelled |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:--------------------|-----------:|-----------:|-------------:|----------------:|-----------:|-----------:|
| msmarco-passage-dev |      15678 |       7697 |         7981 |            4006 |       3538 |        153 |
| beir-nfcorpus       |        323 |        323 |            0 |             220 |         26 |         77 |

labelled 8,020 of 50,000 (16.0%)
